Copy from assignment and modify by TFY
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MillerTFY/6m-data-3.8-Computer-Vision/blob/main/notebooks/assignment_tfy.ipynb)

### Set Runtime type to T4 in Colab

In [1]:
# === Colab-compat setup (no-op when running locally) ===

import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.8-Computer-Vision.git"
    REPO_DIR = "/content/6m-data-3.8-Computer-Vision"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

for _key, _val in [("OMP_NUM_THREADS", "1"), ("MKL_NUM_THREADS", "1"), ("TOKENIZERS_PARALLELISM", "false")]:
    os.environ.setdefault(_key, _val)

In [2]:
# Setup cell: import the libraries we need, lock in randomness for reproducible

import time
import numpy as np                     # numerical arrays
import matplotlib.pyplot as plt        # plotting / showing images
import torch                           # PyTorch — the deep-learning framework
import torch.nn as nn                  # neural-network building blocks (layers, losses)
import torchvision.transforms as T     # image preprocessing pipelines
from torchvision.datasets import CIFAR10
from torchvision.models import resnet18, ResNet18_Weights   # a pretrained model
from torch.utils.data import DataLoader, Subset

torch.set_num_threads(1)   # keep CPU usage to one thread for predictable timing
# Fixing the random "seed" means shuffling, weight init, etc. happen the same way
# every run — so your results are reproducible instead of changing each time.
torch.manual_seed(0)
np.random.seed(0)

# A "transform" is a recipe applied to every image before the model sees it.
# CIFAR-10 is 32×32 colour. Auto-downloads on first use into data/cifar10/.
basic_tf = T.Compose([
    T.ToTensor(),   # convert image to a tensor with pixel values scaled to 0–1
    # Normalize shifts/scales each colour channel so values are centred around 0.
    # These specific numbers are the known mean/std of the CIFAR-10 dataset.
    T.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]),  # CIFAR stats
])

# train=True gives the 50k training images; train=False gives the 10k test images.
train_full = CIFAR10(root='data/cifar10', train=True, download=True, transform=basic_tf)
test_full  = CIFAR10(root='data/cifar10', train=False, download=True, transform=basic_tf)
CLASSES = train_full.classes   # the 10 category names (airplane, cat, ...)

# For runtime budget: subsample training set to 1000/class = 10K. Test stays at 10K for stable eval.
# This helper walks through the dataset and keeps only the first `n` images of
# each class, then stops early once every class has enough — a fast way to shrink data.
def first_n_per_class(dataset, n, n_classes=10):
    counts = {c: 0 for c in range(n_classes)}   # how many we've kept per class
    indices = []
    for i, (_, y) in enumerate(dataset):        # y is the class label of image i
        if counts[y] < n:
            indices.append(i); counts[y] += 1
        if all(v >= n for v in counts.values()): break   # stop once all classes are full
    return Subset(dataset, indices)             # a view onto just those images

train_ds = first_n_per_class(train_full, n=1000)  # 10,000 images
test_ds  = test_full                              # 10,000 images
print(f"Train: {len(train_ds):,}  |  Test: {len(test_ds):,}")
print(f"Classes: {CLASSES}")

 20%|█▉        | 33.8M/170M [14:25<58:13, 39.1kB/s]  


KeyboardInterrupt: 

In [ ]:
# Visualize Data
fig, axes = plt.subplots(2, 5, figsize=(11, 5))   # a 2-row × 5-column grid = 10 slots
for cls_idx, ax in enumerate(axes.flat):          # loop over each grid slot / class
    # Find the first image in the training set whose label matches this class.
    idx = next(i for i, (_, y) in enumerate(train_ds) if y == cls_idx)
    img, _ = train_ds[idx]
    # Tensors store images as (channels, height, width); imshow wants (H, W, channels),
    # so permute reorders the axes before converting to a NumPy array.
    img_disp = img.permute(1, 2, 0).numpy()
    # Earlier we normalised the pixels; here we reverse it (multiply by std, add mean)
    # so the colours look natural again.
    img_disp = img_disp * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    img_disp = np.clip(img_disp, 0, 1)   # keep values in the valid 0–1 range for display
    ax.imshow(img_disp)
    ax.set_title(CLASSES[cls_idx], fontsize=10)
    ax.axis('off')                       # hide the x/y pixel axes
plt.suptitle('CIFAR-10 — one example per class')
plt.tight_layout()                       # stop titles/plots from overlapping
plt.show()

In [ ]:
# Define Model
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),         # -> 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # -> 8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),   
            nn.Linear(128*4*4, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
   
    def forward(self, x):
        return self.classifier(self.features(x))

cnn = CifarCNN()
# Count the learnable numbers (weights) in the model — a rough size measure.
print(f"CifarCNN parameter count: {sum(p.numel() for p in cnn.parameters()):,}")

# Quick shape check: feed in 2 fake images and confirm we get 2 rows of 10 scores.
# If your blanks are correct, this prints  torch.Size([2, 10]).
with torch.no_grad():
    out = cnn(torch.randn(2, 3, 32, 32))   # 2 images, 3 channels, 32x32 pixels
print(f"Output logits shape: {out.shape}")

In [ ]:
# Training

BATCH = 128       # how many images the model processes at once
EPOCHS_A = 8      # how many full passes over the training data
LR = 1e-3         # learning rate: how big a step to take when updating weights

# DataLoaders feed images to the model in batches. shuffle=True on training data
# mixes the order each epoch so the model doesn't memorise the sequence.
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

# Adam is a popular optimiser — it's the algorithm that adjusts the weights.
opt = torch.optim.Adam(cnn.parameters(), lr=LR)
# CrossEntropyLoss measures how wrong the predictions are for a classification task.
crit = nn.CrossEntropyLoss()

# Helper to measure accuracy: fraction of test images the model labels correctly.
def evaluate(model, loader):
    model.eval()                    # switch to evaluation mode (no training behaviour)
    correct, total = 0, 0
    with torch.no_grad():           # don't track gradients — we're only measuring
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)   # pick the class with the highest score
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

print('Training CifarCNN from scratch...')
t0 = time.time()
for epoch in range(1, EPOCHS_A + 1):
    cnn.train()                     # switch to training mode
    running = 0.0                   # accumulate loss to report an average
    for xb, yb in train_loader:     # xb = batch of images, yb = their true labels
        # ┌─ YOUR TASK: the 4 steps of one training update ──────────────────────┐
        # │ This loop is the heart of deep learning. Fill in the 4 blanks in order.│
        # └────────────────────────────────────────────────────────────────────────┘
        ____                        # TODO 1: clear gradients from the last step.   Hint: opt.zero_grad()
        loss = crit(____, yb)       # TODO 2: predict on xb, then compare to labels. Hint: feed cnn(xb)
        ____                        # TODO 3: backpropagate to compute gradients.    Hint: loss.backward()
        ____                        # TODO 4: apply the weight update.               Hint: opt.step()
        running += loss.item() * xb.size(0)
    acc = evaluate(cnn, test_loader)   # check test accuracy after each epoch
    print(f"  epoch {epoch} | loss {running/len(train_ds):.4f} | test acc {acc:.4f}")
scratch_acc = evaluate(cnn, test_loader)   # final accuracy, saved for the Part C comparison
print(f"\n[Part A] CNN from scratch final test accuracy: {scratch_acc:.4f}")
print(f"Training time: {time.time()-t0:.1f}s")
# Quick pass/fail readouts against the assignment targets.
print(f"\nBaseline (≥0.55): {'PASS' if scratch_acc >= 0.55 else 'BELOW — try more epochs'}")
print(f"Stretch  (≥0.65): {'PASS' if scratch_acc >= 0.65 else 'try adding BatchNorm or augmentation'}")

In [ ]:
# Save Model A (the from-scratch CNN) so you can deploy it on Hugging Face later.
torch.save(cnn.state_dict(), 'cifar_cnn.pt')
print('Saved cifar_cnn.pt')

### To be continue......